In [ ]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# Define schemas according to native Statistics Canada layouts
CENSUS_CONFIGS = {
    2021: {
        'usecols': [
            'CENSUS_YEAR', 'DGUID', 'GEO_LEVEL', 'GEO_NAME',
            'CHARACTERISTIC_ID', 'CHARACTERISTIC_NAME',
            'C1_COUNT_TOTAL', 'C2_COUNT_MEN+', 'C3_COUNT_WOMEN+',
            'C10_RATE_TOTAL', 'C11_RATE_MEN+', 'C12_RATE_WOMEN+'
        ],
        'dtypes': {
            'CENSUS_YEAR': np.int64,
            'DGUID': str,
            'GEO_LEVEL': str,
            'GEO_NAME': str,
            'CHARACTERISTIC_ID': np.int64,
            'CHARACTERISTIC_NAME': str,
            'C1_COUNT_TOTAL': np.double,
            'C2_COUNT_MEN+': np.double,
            'C3_COUNT_WOMEN+': np.double,
            'C10_RATE_TOTAL': np.double,
            'C11_RATE_MEN+': np.double,
            'C12_RATE_WOMEN+': np.double,
        },
        'zips': [
            './census_data/98-401-X2021006_eng_CSV.zip',
            './census_data/98-401-X2021010_eng_CSV.zip',
            './census_data/98-401-X2021029_eng_CSV.zip',
            './census_data/98-401-X2021012_eng_CSV.zip'
        ]
    },
    2016: {
        'names': [
            'CENSUS_YEAR', 
            'GEO_CODE',
            'GEO_LEVEL',
            'GEO_NAME',
            'GNR',
            'GNR_LF',
            'DATA_QUALITY_FLAG',
            'ALT_GEO_CODE',
            'CHARACTERISTIC_NAME',
            'CHARACTERISTIC_ID',
            'NOTES',
            'COUNT_TOTAL',
            'COUNT_MALE',
            'COUNT_FEMALE'
        ],
        'cols': [
            'CENSUS_YEAR', 
            'GEO_CODE',
            'GEO_LEVEL',
            'GEO_NAME',
            'CHARACTERISTIC_NAME',
            'CHARACTERISTIC_ID',
            'COUNT_TOTAL',
            'COUNT_MALE',
            'COUNT_FEMALE'            

        ],
        'dtypes': {
            'CENSUS_YEAR': np.int64,
            'GEO_CODE': np.int64,
            'GEO_LEVEL': str,
            'GEO_NAME': str,
            'GNR': np.double,
            'GNR_LF': np.double,
            'DATA_QUALITY_FLAG': str,
            'ALT_GEO_CODE': str,
            'CHARACTERISTIC_NAME': str,
            'CHARACTERISTIC_ID': np.int64,
            'COUNT_TOTAL': np.double,
            'COUNT_MALE': np.double,
            'COUNT_FEMALE': np.double
        },
        'zips': [
            './census_data/98-401-X2016044_eng_CSV.zip',
            './census_data/98-401-X2016045_eng_CSV.zip',
            './census_data/98-401-X2016050_eng_CSV.zip'
        ]
    },
    2011: {
        'names': [
            'GEO_CODE',
            'GEO_NAME',
            'TOPIC',
            'CHARACTERISTIC_NAME',
            'NOTE',
            'COUNT_TOTAL',
            'FLAG_TOTAL',
            'COUNT_MALE',
            'FLAG_MALE',
            'COUNT_FEMALE',
            'FLAG_FEMALE'
        ],
        'cols': [
            'GEO_CODE',
            'GEO_NAME',
            'TOPIC',
            'CHARACTERISTIC_NAME',
            'COUNT_TOTAL',
            'COUNT_MALE',
            'COUNT_FEMALE'
        ],
        'dtypes': {
            'GEO_CODE': np.int64,
            'GEO_NAME': str,      
            'TOPIC': str,
            'CHARACTERISTIC_NAME': str,
            'NOTE': str,
            'COUNT_TOTAL': np.double,
            'FLAG_TOTAL': str,
            'COUNT_MALE': np.double,
            'FLAG_MALE': str,
            'COUNT_FEMALE': np.double,
            'FLAG_FEMALE': str      
        },
        'zips': [
            './census_data/98-316-XWE2011001-101_CSV.zip',
            './census_data/98-316-XWE2011001-301_CSV.zip',
            './census_data/98-316-XWE2011001-401_CSV.zip',
            './census_data/98-316-XWE2011001-501_CSV.zip',
            './census_data/98-316-XWE2011001-701_CSV.zip',
            './census_data/98-316-XWE2011001-1501_CSV.zip',
            './census_data/99-004-XWE2011001-101_CSV.zip',
            './census_data/99-004-XWE2011001-301_CSV.zip',
            './census_data/99-004-XWE2011001-401_CSV.zip',
            './census_data/99-004-XWE2011001-501_CSV.zip',
            './census_data/99-004-XWE2011001-701_CSV.zip'
        ]
    }
}

output_dir = "temp_parquet_data"
os.makedirs(output_dir, exist_ok=True)


# Preprocess 2021 Data #

In [ ]:
def process_year_data(year_config, output_path):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith('.csv'):
                    continue

                print(f"Processing {file}")

                with z.open(file) as f:
                    for chunk in pd.read_csv(
                        f,
                        encoding='latin1',
                        low_memory=False,
                        usecols=year_config['usecols'],
                        dtype=year_config['dtypes'],
                        chunksize=100_000
                    ):

                        table = pa.Table.from_pandas(chunk, preserve_index=False)

                        if writer is None:
                            writer = pq.ParquetWriter(
                                output_path,
                                table.schema,
                                compression='snappy'
                            )

                        writer.write_table(table)
                        print(f"  processed {len(chunk):,} rows")

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

In [ ]:
out_file = os.path.join(
    output_dir,
    "2021_raw_census_data.parquet"
)

process_year_data(CENSUS_CONFIGS[2021], out_file)

: 

# Preprocess 2016 Data #

In [ ]:
# Regex patterns for 2016 dynamic characteristic columns
CHAR_NAME_RE = re.compile(r'^DIM:\s*Profile of.*$', re.I)
CHAR_ID_RE = re.compile(r'^Member ID:\s*Profile of.*$', re.I)

def process_2016_data(year_config, output_path):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue        

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith('.csv'):
                    continue

                if file == 'Geo_starting_row_CSV.csv':
                    continue                

                print(f"Processing {file}")
                with z.open(file) as f:
                    for chunk in pd.read_csv(
                        f,
                        encoding='utf-8-sig',
                        low_memory=False,
                        header=0,
                        names=year_config['names'],
                        dtype=year_config['dtypes'],
                        na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                        chunksize=100_000
                    ):
                        chunk = chunk[year_config['cols']]

                        table = pa.Table.from_pandas(chunk, preserve_index=False)

                        if writer is None:
                            writer = pq.ParquetWriter(
                                output_path,
                                table.schema,
                                compression='snappy'
                            )

                        writer.write_table(table)
                        print(f"  processed {len(chunk):,} rows")

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")


In [ ]:
out_file = os.path.join(
    output_dir,
    "2016_raw_census_data.parquet"
)

process_2016_data(CENSUS_CONFIGS[2016], out_file)

# Preprocess 2011 Data #

In [ ]:

def process_2011_data(year_config, output_path):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                # the zip file contains many .csv files but only the one with a matching name is useful.
                test_path = zip_path[14:-8]

                if (file != test_path + ".CSV"):
                    continue

                print(f"Processing {file}")

                with z.open(file) as f:
                    for chunk in pd.read_csv(
                        f,
                        encoding='latin1',
                        low_memory=False,
                        header=0,
                        na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                        chunksize=100_00
                    ):
                        chunk = chunk[year_config['cols']]

                        table = pa.Table.from_pandas(chunk, preserve_index=False)

                        if writer is None:
                            writer = pq.ParquetWriter(
                                output_path,
                                table.schema,
                                compression='snappy'
                            )

                        writer.write_table(table)
                        print(f"  processed {len(chunk):,} rows")

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")